# Fuego contra Fuego


Instalamos, cargamos y seteamos el entorno

In [42]:
%pip install optuna


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [43]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

from sklearn.tree import DecisionTreeClassifier, plot_tree,  _tree
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ShuffleSplit, StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

from joblib import Parallel, delayed

import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice, plot_contour

from time import time

import pickle


In [44]:
base_path = '/Users/manuelmarasco/Library/CloudStorage/GoogleDrive-hola@manuelmarasco.com/Mi unidad/DMEyF/2026/notebooks/'
dataset_path = base_path + 'data/'
modelos_path = base_path + 'modelos/'
db_path = base_path + 'data/'
dataset_file = 'competencia_01.csv'

ganancia_acierto = 1_072_500
costo_estimulo = 27_500

mes_train = 202103
mes_test = 202105

# agregue sus semillas
semilla = 17
semillas = np.random.choice(1000000, size=50, replace=False).tolist()

data = pd.read_csv(dataset_path + dataset_file)

/var/folders/xn/fmllb0ws2fbdtxppprq7r2l00000gn/T/ipykernel_12732/2551575851.py:17: DtypeWarning: Columns (155) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(dataset_path + dataset_file)


Seguimos trabajando con Marzo como entrenamiento y Mayo como test.

In [45]:
X = data[data['foto_mes'] == mes_train]
y = X['clase_ternaria']
X = X.drop(columns=['clase_ternaria'])

In [46]:
X_futuro = data[data['foto_mes'] == mes_test]
y_futuro = X_futuro['clase_ternaria']
X_futuro = X_futuro.drop(columns=['clase_ternaria'])

Y variaremos la forma de la función de ganancia, para poder ser utilizada de una forma más genérica.

In [47]:
def ganancia_prob(y_hat, y, prop=1, class_index=1, threshold=0.025):
  @np.vectorize
  def ganancia_row(predicted, actual, threshold=0.025):
    return  (predicted >= threshold) * (ganancia_acierto if actual == "BAJA+2" else -costo_estimulo)

  return ganancia_row(y_hat[:,class_index], y).sum() / prop



Entrenemos y recordemos cuanto nos daba la ganancia en **Mayo**

Hablemos de ensemables, qué son?

* Un ensemble de modelos es una técnica donde se combinan múltiples modelos individuales para mejorar la precisión y robustez de las predicciones. La idea es que al combinar varios modelos, se pueden aprovechar las fortalezas de cada uno y reducir la posibilidad de errores que podría cometer un único modelo.

**Tipos de ensemble**:
 * **Bagging (Bootstrap Aggregating)**: Consiste en entrenar varios modelos base en diferentes subconjuntos del conjunto de datos de entrenamiento obtenidos mediante técnicas de remuestreo como el bootstrap y luego promediar sus predicciones. Ejemplo: **Random Forest**
 * **Boosting**: En esta técnica, los modelos se entrenan de manera secuencial. Cada modelo intenta corregir los errores cometidos por el modelo anterior. Ejemplo: AdaBoost y **Gradient Boosting**.
 * **Stacking**: En el stacking, se entrenan varios modelos y se combinan usando un "modelo meta". Las predicciones de los modelos base sirven como features para entrenar este modelo meta, que produce la predicción final.

* **Ventajas de usar ensemble de modelos**:
 * **Mejor rendimiento**: Al combinar modelos, generalmente se mejora la precisión en comparación con un solo modelo.
 * **Robustez**: Al integrar diferentes modelos, se mitiga el riesgo de que los errores de un modelo individual afecten gravemente la predicción final.




Pongamos foco en el **Random Forest**

Es un algoritmo de aprendizaje automático que funciona creando un conjunto de árboles de decisión. Para la creación de **árboles distintos** utiliza una técnica llamada bagging para crear múltiples subconjuntos del conjunto de datos de entrenamiento. Cada subconjunto se genera seleccionando al azar muestras del conjunto de datos original con reemplazo. No usa la totalidad de los datos de entrenamiento para cada conjunto. Los datos que quedan fueran son conocidos como **Out of Bag (oob)**

Para cada subconjunto, se construye un árbol de decisión. Sin embargo en cada nodo del árbol, **Random Forest** selecciona de forma aleatoria un grupo de variables y ajusta el árbol con esas variables. Este proceso ayuda a crear árboles que son menos correlacionados entre sí.

Cada árbol en el bosque se entrena de manera independiente usando su respectivo subconjunto de datos. Lueago, para una nueva observación, cada árbol realiza una predicción. El Random Forest luego combina las predicciones de todos los árboles para hacer una predicción final, devolviendo el promedio de las probabilidades de cada árbol individual.



Como desde la clase pasada solo personas más inteligentes, no vamos a empezar a probar **Random Forest** simples. Vamos a parametrizarlo desde el vamos.

Primero vamos a entender algunas limitaciones de la implementación:

El **Random Forest** no soporta nulos! Shame on you sklearn!.

Vamos a tener que imputar los datos. Discutamos entre todos forma de imputar los datos, mientras para salir del paso usamos la peor de todas.

In [48]:
imp_median = SimpleImputer(missing_values=np.nan, strategy='median')
Xi = imp_median.fit_transform(X)
Xif = imp_median.fit_transform(X_futuro)

Los parámetros que se pueden ajustar en el **rf** son

1. **n_estimators**: Número de árboles en el bosque.
2. **max_depth**: Profundidad máxima de los árboles.
3. **min_samples_split**: Número mínimo de muestras requeridas para dividir un nodo interno.
4. **min_samples_leaf**: Número mínimo de muestras requeridas para estar en un nodo hoja.
5. **max_features**: Número de features a usar en cada árbol. **sqrt** es una elección histórica.
6. **max_leaf_nodes**: Número máximo de nodos hoja en cada árbol.
7. **oob_score**: Indica si se usa la muestra fuera de bolsa (out-of-bag) para estimar la calidad del modelo. Para evitar hacer un **montecarlo-cross-validation** que se toma su tiempo, usaremos esta opción para buscar el mejor modelo. No es la mejor opción. Pero no es tan mala.
8. **n_jobs**: Siempre -1, para que use todos los cores presentes en 9. **max_samples**: Fracción de los samples.

Finalmente nuestra función de optimización queda la siguiente forma:

In [ ]:
# def objective(trial):
#     max_depth = trial.suggest_int('max_depth', 2, 32)
#     min_samples_split = trial.suggest_int('min_samples_split', 2, 2000)
#     min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 200)
#     max_features = trial.suggest_float('max_features', 0.05, 0.7)

#     model = RandomForestClassifier(
#         n_estimators=100,
#         max_depth=max_depth,
#         min_samples_split=min_samples_split,
#         min_samples_leaf=min_samples_leaf,
#         max_features=max_features,
#         max_samples=0.7,
#         random_state=semillas[0],
#         n_jobs=-1,
#         oob_score=True
#     )

#     model.fit(Xi, y)

#     return ganancia_prob(model.oob_decision_function_, y)

# storage_name = "sqlite:///" + db_path + "optimization_rf.db"
# study_name = "exp_401_random-forest-opt"

# study = optuna.create_study(
#     direction="maximize",
#     study_name=study_name,
#     storage=storage_name,
#     load_if_exists=True,
# )



[I 2026-09-07 14:20:12,244] Using an existing study with name 'exp_401_random-forest-opt' instead of creating a new one.


In [52]:
study.optimize(objective, n_trials=100)

[I 2026-09-07 14:20:48,032] Trial 243 finished with value: 346307500.0 and parameters: {'max_depth': 17, 'min_samples_split': 395, 'min_samples_leaf': 53, 'max_features': 0.6554986965330545}. Best is trial 233 with value: 356372500.0.
[I 2026-09-07 14:21:17,892] Trial 244 finished with value: 330522500.0 and parameters: {'max_depth': 17, 'min_samples_split': 382, 'min_samples_leaf': 52, 'max_features': 0.6571358876168024}. Best is trial 233 with value: 356372500.0.
[I 2026-09-07 14:21:50,505] Trial 245 finished with value: 347132500.0 and parameters: {'max_depth': 17, 'min_samples_split': 365, 'min_samples_leaf': 51, 'max_features': 0.6466681857048291}. Best is trial 233 with value: 356372500.0.
[I 2026-09-07 14:22:21,211] Trial 246 finished with value: 344575000.0 and parameters: {'max_depth': 17, 'min_samples_split': 461, 'min_samples_leaf': 51, 'max_features': 0.644150529321394}. Best is trial 233 with value: 356372500.0.
[I 2026-09-07 14:22:49,561] Trial 247 finished with value: 34

Best is trial 233 with value: 356372500.

Tarea: 

Mejora 2 Promediar la ganancia OOB sobre varias semillas (evita que Optuna "aprenda" hiperparámetros que solo funcionan bien con semillas[0])

In [54]:
def objective2(trial):
    max_depth = trial.suggest_int('max_depth', 2, 32)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 2000)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 200)
    max_features = trial.suggest_float('max_features', 0.05, 0.7)

    n_semillas_objetivo = 3
    ganancias = []

    for s in semillas[:n_semillas_objetivo]:
        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            max_samples=0.7,
            random_state=s,
            n_jobs=-1,
            oob_score=True
        )
        model.fit(Xi, y)
        ganancias.append(ganancia_prob(model.oob_decision_function_, y))

    return np.mean(ganancias)

storage_name = "sqlite:///" + db_path + "optimization2_rf.db"
study_name = "exp_401_random-forest-opt-multiseed"

study2 = optuna.create_study(
    direction="maximize",
    study_name=study_name,
    storage=storage_name,
    load_if_exists=True,
)

[I 2026-09-07 14:55:31,707] Using an existing study with name 'exp_401_random-forest-opt-multiseed' instead of creating a new one.


Mejora 3 (en paralelo)

El bloque 2 no toca la metodología de evaluación (como el bloque 1), sino el espacio de búsqueda: qué hiperparámetros explora Optuna y en qué rangos. Este es el código:


def objective(trial):
    max_depth = trial.suggest_int('max_depth', 2, 32)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 2000, log=True)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 60, log=True)
    max_features = trial.suggest_float('max_features', 0.05, 0.7)
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy', 'log_loss'])
    class_weight = trial.suggest_categorical('class_weight', [None, 'balanced', 'balanced_subsample'])
    ...
Tres cambios concretos respecto al original:

1. Agrega class_weight

Random Forest, por default, trata todas las filas por igual al decidir cómo partir cada nodo del árbol. El problema: BAJA+2 es una clase minoritaria (poquísimos clientes se dan de baja comparado con los que no). Si todas las filas pesan igual, al árbol "le conviene" ignorar un poco esa clase rara, porque acertar en la mayoría (los que no se van) ya le da buena pureza en el nodo, aunque nunca aprenda a distinguir bien a los que sí se van — que es justo lo que te importa para la ganancia del negocio.

class_weight='balanced' le dice a sklearn: "pesá más los errores en la clase rara, en proporción inversa a cuántas veces aparece". Así el árbol se ve forzado a prestarle más atención a BAJA+2 al elegir sus cortes. 'balanced_subsample' hace lo mismo pero recalculando el peso en cada muestra bootstrap (por árbol) en vez de una sola vez sobre todo el dataset. Agregarlo como hiperparámetro deja que Optuna decida solo si conviene pesar o no (None = sin peso, el comportamiento actual).

2. Agrega criterion

Es la fórmula que usa el árbol para medir qué tan "pura" queda cada partición al elegir dónde cortar (gini es el default de sklearn, entropy/log_loss son alternativas). Con clases desbalanceadas, a veces una fórmula distinta arma cortes ligeramente distintos que ayudan a aislar mejor la clase rara. Es barato probarlo, así que se lo dejamos elegir a Optuna.

3. Achica min_samples_leaf (de 1-200 a 1-60) y usa escala logarítmica en ambos rangos

El rango original permitía hasta 200 muestras mínimas por hoja. Si BAJA+2 es muy rara, exigir 200 muestras por hoja puede obligar al árbol a mezclar esos pocos casos raros con un montón de "no se va", diluyendo la señal. Bajar el techo a 60 le da a Optuna más margen para probar hojas chicas que puedan aislar mejor los clusters de clientes que sí se van.
log=True cambia cómo Optuna muestrea dentro del rango: en vez de repartir intentos uniformemente entre 1 y 60 (desperdiciando pruebas comparando, digamos, 45 vs 50, que se comportan casi igual), concentra más intentos en valores chicos (1, 2, 3, 5, 10...) donde el impacto de cambiar el número es mucho mayor. Es más eficiente la búsqueda.
En resumen: el bloque 1 te da una medición más confiable de "qué tan buena es una combinación de hiperparámetros"; el bloque 2 amplía y afina cuáles combinaciones se prueban, apuntando específicamente al problema de clase desbalanceada que viste en los resultados de la clase.



In [70]:
def objective3(trial):
    max_depth = trial.suggest_int('max_depth', 2, 32)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 2000, log=True)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 60, log=True)
    max_features = trial.suggest_float('max_features', 0.05, 0.7)
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy', 'log_loss'])
    class_weight = trial.suggest_categorical('class_weight', [None, 'balanced', 'balanced_subsample'])

    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        criterion=criterion,
        class_weight=class_weight,
        max_samples=0.7,
        random_state=semillas[0],
        n_jobs=-1,
        oob_score=True
    )

    model.fit(Xi, y)

    return ganancia_prob(model.oob_decision_function_, y)

In [71]:
storage_name = "sqlite:///" + db_path + "optimization_rf.db"
study_name2 = "exp_401_random-forest-opt-searchspace"

study3 = optuna.create_study(
    direction="maximize",
    study_name=study_name2,
    storage=storage_name,
    load_if_exists=True,
)

[I 2026-09-07 16:42:49,848] A new study created in RDB with name: exp_401_random-forest-opt-searchspace


In [55]:
study2.optimize(objective2, n_trials=100)

[I 2026-09-07 14:55:53,198] Trial 0 finished with value: 317496666.6666667 and parameters: {'max_depth': 32, 'min_samples_split': 708, 'min_samples_leaf': 162, 'max_features': 0.18409492193373514}. Best is trial 0 with value: 317496666.6666667.
[I 2026-09-07 14:56:31,520] Trial 1 finished with value: 315333333.3333333 and parameters: {'max_depth': 10, 'min_samples_split': 1447, 'min_samples_leaf': 7, 'max_features': 0.42694688177882933}. Best is trial 0 with value: 317496666.6666667.
[I 2026-09-07 14:57:45,133] Trial 2 finished with value: 313371666.6666667 and parameters: {'max_depth': 13, 'min_samples_split': 1440, 'min_samples_leaf': 4, 'max_features': 0.642863694339371}. Best is trial 0 with value: 317496666.6666667.
[I 2026-09-07 14:58:19,015] Trial 3 finished with value: 321795833.3333333 and parameters: {'max_depth': 6, 'min_samples_split': 1283, 'min_samples_leaf': 155, 'max_features': 0.6391527843758886}. Best is trial 3 with value: 321795833.3333333.
[I 2026-09-07 14:58:48,41

In [72]:
study3.optimize(objective3, n_trials=100)

[I 2026-09-07 16:43:26,044] Trial 0 finished with value: 111045000.0 and parameters: {'max_depth': 25, 'min_samples_split': 13, 'min_samples_leaf': 1, 'max_features': 0.6686587709811007, 'criterion': 'entropy', 'class_weight': 'balanced'}. Best is trial 0 with value: 111045000.0.
[I 2026-09-07 16:43:31,278] Trial 1 finished with value: 291582500.0 and parameters: {'max_depth': 4, 'min_samples_split': 2, 'min_samples_leaf': 42, 'max_features': 0.343011939765005, 'criterion': 'gini', 'class_weight': None}. Best is trial 1 with value: 291582500.0.
[I 2026-09-07 16:43:47,219] Trial 2 finished with value: 352082500.0 and parameters: {'max_depth': 11, 'min_samples_split': 29, 'min_samples_leaf': 29, 'max_features': 0.5983917329186031, 'criterion': 'entropy', 'class_weight': None}. Best is trial 2 with value: 352082500.0.
[I 2026-09-07 16:44:00,036] Trial 3 finished with value: 352770000.0 and parameters: {'max_depth': 20, 'min_samples_split': 778, 'min_samples_leaf': 2, 'max_features': 0.455

Exploramos como fue la búsqueda de parámetros

In [16]:
import sys, nbformat
print(sys.executable)
print(nbformat.__version__)

/Library/Developer/CommandLineTools/usr/bin/python3
5.10.4


In [10]:
optuna.visualization.plot_optimization_history(study)

In [56]:
optuna.visualization.plot_optimization_history(study2)

In [11]:
plot_param_importances(study)

In [57]:
plot_param_importances(study2)

In [12]:
plot_slice(study)

In [58]:
plot_slice(study2)

In [13]:
plot_contour(study)

In [59]:
plot_contour(study2)

In [14]:
plot_contour(study, params=["max_depth", "min_samples_split"])

In [60]:
plot_contour(study2, params=["max_depth", "min_samples_split"])

Ajustamos el mejor modelo a todo el conjunto de datos



In [15]:
model_rf = RandomForestClassifier(
        n_estimators=100,
        **study.best_params,
        max_samples=0.7,
        random_state=semillas[0],
        n_jobs=-1,
        oob_score=True
    )

model_rf.fit(Xi, y)


RandomForestClassifier(max_depth=15, max_features=0.6245837962050735,
                       max_samples=0.7, min_samples_leaf=53,
                       min_samples_split=339, n_jobs=-1, oob_score=True,
                       random_state=713929)

In [65]:
model_rf2 = RandomForestClassifier(
        n_estimators=100,
        **study2.best_params,
        max_samples=0.7,
        random_state=semillas[0],
        n_jobs=-1,
        oob_score=True
    )

model_rf2.fit(Xi, y)

RandomForestClassifier(max_depth=13, max_features=0.41542696991016825,
                       max_samples=0.7, min_samples_leaf=53,
                       min_samples_split=50, n_jobs=-1, oob_score=True,
                       random_state=808840)

In [73]:
model_rf3 = RandomForestClassifier(
        n_estimators=100,
        **study3.best_params,
        max_samples=0.7,
        random_state=semillas[0],
        n_jobs=-1,
        oob_score=True
    )

model_rf3.fit(Xi, y)

RandomForestClassifier(criterion='entropy', max_depth=12,
                       max_features=0.5423023481240334, max_samples=0.7,
                       min_samples_leaf=27, min_samples_split=210, n_jobs=-1,
                       oob_score=True, random_state=808840)

Guardamos el modelo, para no tener que optimizar cada vez que lo queramos usar.

In [16]:
filename = modelos_path + 'exp_206_random_forest_model_100.sav'
pickle.dump(model_rf, open(filename, 'wb'))


In [66]:
filename = modelos_path + 'exp_71_random_forest_model_100.sav'
pickle.dump(model_rf2, open(filename, 'wb'))


In [74]:
filename = modelos_path + 'exp_91_random_forest_model_100.sav'
pickle.dump(model_rf3, open(filename, 'wb'))

Y lo cargamos para utilizarlo

In [17]:
model_rf = pickle.load(open(filename, 'rb'))
model_rf

RandomForestClassifier(max_depth=15, max_features=0.6245837962050735,
                       max_samples=0.7, min_samples_leaf=53,
                       min_samples_split=339, n_jobs=-1, oob_score=True,
                       random_state=713929)

In [67]:
model_rf2 = pickle.load(open(filename, 'rb'))
model_rf2

RandomForestClassifier(max_depth=13, max_features=0.41542696991016825,
                       max_samples=0.7, min_samples_leaf=53,
                       min_samples_split=50, n_jobs=-1, oob_score=True,
                       random_state=808840)

In [75]:
model_rf3 = pickle.load(open(filename, 'rb'))
model_rf3

RandomForestClassifier(criterion='entropy', max_depth=12,
                       max_features=0.5423023481240334, max_samples=0.7,
                       min_samples_leaf=27, min_samples_split=210, n_jobs=-1,
                       oob_score=True, random_state=808840)

Ahora vamos medir su ganancia sobre el dataset de **Mayo**

In [18]:
y_pred_rf = model_rf.predict_proba(Xif)
ganancias_rf = ganancia_prob(y_pred_rf, y_futuro)
print(f"Ganancia de modelo RF: {ganancias_rf}")

Ganancia de modelo RF: 253137500.0


In [68]:
y_pred_rf2 = model_rf2.predict_proba(Xif)
ganancias_rf2 = ganancia_prob(y_pred_rf2, y_futuro)
print(f"Ganancia de modelo RF: {ganancias_rf2}")

Ganancia de modelo RF: 254732500.0


In [76]:
y_pred_rf3 = model_rf3.predict_proba(Xif)
ganancias_rf3 = ganancia_prob(y_pred_rf3, y_futuro)
print(f"Ganancia de modelo RF: {ganancias_rf3}")

Ganancia de modelo RF: 245327500.0


Frente a los 208.010.000 de mi mejor árbol es algo sustancialmente mejor.

Igualmente usamos poquitos estimadores. Tan solo unos 100. Suficientes como para que la optimización no tarde una eternidad, pero es muy poco. Deberíamos sumar unos cuantos más.

In [ ]:
model_rf_1000 = RandomForestClassifier(
        n_estimators=1000,
        **study.best_params,
        max_samples=0.7,
        random_state=semillas[0],
        n_jobs=-1,
        oob_score=True
    )

model_rf_1000.fit(Xi, y)

filename_rf_1000 = modelos_path + 'exp_206_random_forest_model_1000.sav'
pickle.dump(model_rf, open(filename_rf_1000, 'wb'))


In [69]:
model_rf_1000_2 = RandomForestClassifier(
        n_estimators=1000,
        **study2.best_params,
        max_samples=0.7,
        random_state=semillas[0],
        n_jobs=-1,
        oob_score=True
    )

model_rf_1000_2.fit(Xi, y)

filename_rf_1000_2 = modelos_path + 'exp_71_random_forest_model_1000.sav'
pickle.dump(model_rf2, open(filename_rf_1000_2, 'wb'))

In [78]:
model_rf_1000_3 = RandomForestClassifier(
        n_estimators=1000,
        **study3.best_params,
        max_samples=0.7,
        random_state=semillas[0],
        n_jobs=-1,
        oob_score=True
    )

model_rf_1000_3.fit(Xi, y)

filename_rf_1000_3 = modelos_path + 'exp_91_random_forest_model_1000.sav'
pickle.dump(model_rf3, open(filename_rf_1000_3, 'wb'))

In [ ]:
model_rf_1000 = pickle.load(open(filename_rf_1000, 'rb'))
model_rf_1000

In [ ]:
model_rf_1000_2 = pickle.load(open(filename_rf_1000_2, 'rb'))
model_rf_1000_2

In [ ]:
model_rf_1000_3 = pickle.load(open(filename_rf_1000_3, 'rb'))
model_rf_1000_3

Veamos si sumar 10 veces más estimadores tuvo algún efecto

In [ ]:
y_pred_rf = model_rf_1000.predict_proba(Xif)
ganancias_rf = ganancia_prob(y_pred_rf, y_futuro)
print(f"Ganancia de modelo RF 1000: {ganancias_rf}")

Ganancia de modelo RF 1000: 253137500.0


Mejoró un poco! Cada peso vale.



Por último evaluamos cuales son las variables más importantes del modelo. ¿Habrá influido la imputación?

In [20]:
importances = model_rf.feature_importances_

features = X.columns
feat_importances = pd.DataFrame({'feature': features, 'importance': importances})
feat_importances = feat_importances.sort_values('importance', ascending=False)

feat_importances.head(25)


,feature,importance
107,ctrx_quarter,0.219703
22,mcuentas_saldo,0.086965
33,mprestamos_personales,0.064014
18,mcaja_ahorro,0.059259
11,mpasivos_margen,0.042471
133,Visa_status,0.033728
99,ccajas_consultas,0.033291
16,mcuenta_corriente,0.032875
8,mrentabilidad_annual,0.032861
21,cdescubierto_preacordado,0.031803


Vamos a hacer un par de experimentos, que pasa si sacamos la variable más importante?

In [21]:
most_important_feature = feat_importances.iloc[0]['feature']

X_df = pd.DataFrame(Xi, columns=X.columns)
X_futuro_df = pd.DataFrame(Xif, columns=X_futuro.columns)

X_no_best_feature = X_df.drop(columns=[most_important_feature])
X_futuro_no_best_feature = X_futuro_df.drop(columns=[most_important_feature])

model_rf_no_best_feature = RandomForestClassifier(
    n_estimators=100,
    **study.best_params,
    max_samples=0.7,
    random_state=semillas[0],
    n_jobs=-1,
    oob_score=True
)

print(f"Entrenando modelo RF sin '{most_important_feature}'...")
model_rf_no_best_feature.fit(X_no_best_feature, y)

y_pred_rf_no_best_feature = model_rf_no_best_feature.predict_proba(X_futuro_no_best_feature)

ganancia_rf_no_best_feature = ganancia_prob(y_pred_rf_no_best_feature, y_futuro)

print(f"Ganancia del modelo RF sin '{most_important_feature}': {ganancia_rf_no_best_feature}")
print(f"Ganancia del modelo RF original: {ganancias_rf}")

Entrenando modelo RF sin 'ctrx_quarter'...
Ganancia del modelo RF sin 'ctrx_quarter': 245767500.0
Ganancia del modelo RF original: 253137500.0


¿Qué paso? ¿Se ha derrumbado las ganancias frente a la ausencia del pilar de los árboles?

Una prueba más, que pasa si sumamos variables que son ruido puro.

In [22]:
X_df_from_Xi = pd.DataFrame(Xi, columns=X.columns)

np.random.seed(semilla)

random_features_train = pd.DataFrame({
    f'random_feature_{i}': np.random.rand(len(X_df_from_Xi)) for i in range(1, 6)
})

X_with_random_features = pd.concat([X_df_from_Xi, random_features_train], axis=1)

model_rf_new_features = RandomForestClassifier(
    n_estimators=100,
    **study.best_params,
    max_samples=0.7,
    random_state=semillas[0],
    n_jobs=-1,
    oob_score=True
)

model_rf_new_features.fit(X_with_random_features, y)

importances_new_features = model_rf_new_features.feature_importances_

features_new_features = X_with_random_features.columns
feat_importances_new_features = pd.DataFrame({'feature': features_new_features, 'importance': importances_new_features})
feat_importances_new_features = feat_importances_new_features.sort_values('importance', ascending=False)

display(feat_importances_new_features.head(25))

,feature,importance
107,ctrx_quarter,0.228494
22,mcuentas_saldo,0.094135
33,mprestamos_personales,0.061438
11,mpasivos_margen,0.056584
18,mcaja_ahorro,0.041629
99,ccajas_consultas,0.035760
21,cdescubierto_preacordado,0.031391
8,mrentabilidad_annual,0.030823
16,mcuenta_corriente,0.027401
133,Visa_status,0.026903


Y una más, cual es el área bajo la curva en train?

In [30]:
from sklearn.metrics import roc_auc_score

y_pred_rf_train = model_rf_1000.predict_proba(Xi)

ganancia_rf_train = ganancia_prob(y_pred_rf_train, y)
print(f"Ganancia en el conjunto de entrenamiento: {ganancia_rf_train}")
print(f"Ganancia en el conjunto de Mayo: {ganancias_rf}")

y_train_numeric = (y == 'BAJA+2').astype(int)
y_futuro_numeric = (y_futuro == 'BAJA+2').astype(int)

auc_train = roc_auc_score(y_train_numeric, y_pred_rf_train[:, 1])
print(f"Área bajo la curva (AUC) en el conjunto de entrenamiento: {auc_train}")

auc_mayo = roc_auc_score(y_futuro_numeric, y_pred_rf[:, 1])
print(f"Área bajo la curva (AUC) en el conjunto de Mayo: {auc_mayo}")

Ganancia en el conjunto de entrenamiento: 523600000.0
Ganancia en el conjunto de Mayo: 251570000.0
Área bajo la curva (AUC) en el conjunto de entrenamiento: 0.9606595035198221
Área bajo la curva (AUC) en el conjunto de Mayo: 0.8839194909782766


Cuanta diferencia entre conjuntos. Seguramente debemos descartar este algoritmo, ¿o no?

Pregunta:
* **¿Cómo sabe que un random forest es superior a otro?**
* **¿Cómo supera el random forest presentado?**




¿Cómo sabe que un random forest es superior a otro?

No alcanza con mirar el oob_score/ganancia OOB que usa Optuna en objective() (cell-16) para elegir hiperparámetros — el OOB es una proxy barata (evita hacer Monte Carlo CV) pero se calcula sobre la misma foto de Marzo, con la misma distribución de la que salió el modelo. La comparación real es:

Ganancia en un conjunto verdaderamente fuera de muestra y fuera de tiempo — acá Mayo (mes_test), que es justamente lo que hiciste en cell-31/36: comparar 251.570.000 (100 árboles) vs 253.137.500 (1000 árboles). Como el negocio es temporal (predecís el futuro con el pasado), la comparación válida es train (Marzo) vs. un mes posterior, no un split aleatorio del mismo mes.
Que la ganancia no dependa de una sola semilla — un RF puede "ganarle" a otro por azar en el bootstrap/random_state. Por eso el notebook arma una lista semillas de 50 valores: correr cada configuración candidata con varias semillas y comparar la ganancia promedio (o su dispersión) es más confiable que comparar un único run.
Que no haya overfitting severo — en cell-45 ves AUC 0.96 en train vs 0.88 en Mayo, y ganancia 523.6M en train vs 251.5M en Mayo. Un RF que "gana" en train pero se derrumba más en test no es mejor, aunque su métrica de training sea más alta.
¿Cómo supera el random forest presentado a lo anterior?

En cell-32 el notebook menciona que el mejor árbol individual (DecisionTreeClassifier) daba 208.010.000 de ganancia en Mayo. El RF con 100 árboles ya lo supera (251.570.000) y con 1000 sube un poco más (253.137.500). La razón no es que cada árbol individual del bosque sea mejor que el árbol óptimo — de hecho probablemente cada uno es peor, porque se entrena con bootstrap (menos datos efectivos) y con un subconjunto aleatorio de variables en cada split (max_features). La mejora viene de:

Bagging: promediar muchos árboles entrenados en distintos subconjuntos reduce la varianza (un árbol solo se sobreajusta a particularidades de la muestra; el promedio de 100-1000 árboles "cancela" ese ruido).
Decorrelación vía max_features: al forzar que cada split solo considere un subconjunto aleatorio de variables, los árboles del bosque se parecen menos entre sí, lo que hace que el promedio sea más efectivo que promediar árboles muy correlacionados (que aportarían poco más que un árbol solo).
Y el salto de 100 a 1000 estimadores (253.1M vs 251.5M) es chico porque el bagging tiene rendimientos decrecientes: la reducción de varianza por agregar más árboles converge rápido — pasado cierto punto, sumar árboles ya no decorrelaciona nada nuevo, solo promedia más finamente el mismo conjunto de patrones.

## Tarea:

* Mejore la parametrización del **rf**

1. Promediar la ganancia OOB sobre varias semillas (evita que Optuna "aprenda" hiperparámetros que solo funcionan bien con semillas[0]) El código original entrenaba un solo Random Forest por cada combinación de hiperparámetros que probaba Optuna, siempre con la misma semilla (semillas[0]). El problema: quizás esa combinación de hiperparámetros parece buena o mala un poco por casualidad, porque le tocó esa semilla en particular (la semilla determina qué filas caen en el bootstrap de cada árbol).

El bloque 1 en vez de eso, para cada combinación de hiperparámetros que Optuna quiere probar:

Entrena 3 Random Forest distintos (mismos hiperparámetros, pero 3 semillas diferentes).
Calcula la ganancia OOB de cada uno de los 3.
Devuelve el promedio de esas 3 ganancias como la "nota" de esa combinación de hiperparámetros.
Así, si una combinación de hiperparámetros da buenos resultados solo por azar con una semilla puntual, el promedio con las otras 2 semillas lo va a corregir. Es una nota más confiable, a costa de tardar 3 veces más por cada combinación que prueba Optuna.

¿En qué orden va todo?

El bloque 1 reemplaza el contenido de la celda 16 (ya lo dejé insertado ahí). No es un paso adicional, es la nueva versión de esa celda. De ahí en más, el notebook sigue exactamente el mismo orden que tenía antes:

Celda 16 (con el bloque 1 ya adentro) → define objective y crea el study
Celda 17 → study.optimize(...), ahora usando el objetivo de 3 semillas
Celdas 19-23 → gráficos de Optuna
Celda 25 en adelante → entrenar el modelo final con study.best_params